# Initial design thoughts


## Given

- `Irradiance` object that contains
    - One `Resin` object
        - `ResinConstituentAbsorber` objects
            - `AbsorberSpectrum` objects with w/w concentrations for absorbers and photoinitiators
        - `ResinConstituentMonomer` object(s)
            - `Monomer` object(s) with %monomer concentration(s)
    - One `SourceSpectrum` object
    

## Calculate

- D_n(z)
    - Array
    - Plot
- Model 1: h_a
- Model 2: a, b
- a, b, c Model


# Code

## Imports

In [1]:
import json
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
%matplotlib widget

from scipy.optimize import curve_fit
import scipy.integrate as integrate

import spectra.data as data
from spectra.resin import ResinConstituentAbsorber, ResinConstituentMonomer, Resin
from spectra.irradiance import Irradiance
from spectra.utilities import ArrayParams

## Models

In [2]:
def ha_model(z, b):
    return np.exp(-z / b)


def a_b_model(z, a, b):
    return a * np.exp(-z / b) + (1 - a)


def a_b_c_model(z, a, b, c):
    return a * np.exp(-z / b) + c


def fit_ha_model(z, dnorm):
    param_bounds = ([0.1], [np.inf])
    popt, pcov = curve_fit(ha_model, z, dnorm, bounds=param_bounds)
    return popt, pcov


def fit_a_b_model(z, dnorm):
    param_bounds = ([0., 0.1], [1., np.inf])
    popt, pcov = curve_fit(a_b_model, z, dnorm, bounds=param_bounds)
    return popt, pcov


def fit_a_b_c_model(z, dnorm):
    param_bounds = ([0., 0.1, 0.], [1., np.inf, 1.])
    popt, pcov = curve_fit(a_b_c_model, z, dnorm, bounds=param_bounds)
    return popt, pcov


## NormalizedDose

In [6]:
class NormalizedDose:
    
    def __init__(self, *, irradiance, z_array_params):
        
        assert isinstance(irradiance, Irradiance)
        assert isinstance(z_array_params, ArrayParams)
        
        # Store inputs
        self.irradiance = irradiance
        self.z_array_params = z_array_params
        
        # Create array of z values in microns
        self.z_um = np.linspace(*z_array_params)
        
        # Pre-calculate source integral over wavelengths
        self.source_integral = integrate.simps(self.irradiance(0), self.irradiance.wavelengths)
        
        # Calculate normalized dose as a function of z
        self.normalized_dose = self._calc_normalized_dose()
        
        # Fit normalized dose to models
        self.ha_model_params = self._fit_to_ha_model()
        self.a_b_model_params = self._fit_to_a_b_model()
        self.a_b_c_model_params = self._fit_to_a_b_c_model()
        
        # Calculate normalized dose for each model
        self.ha_model = ha_model(self.z_um, self.ha_model_params['ha'])
        self.a_b_model = a_b_model(self.z_um,
                                   self.a_b_model_params['a'],
                                   self.a_b_model_params['b']
                                  )
        self.a_b_c_model = a_b_c_model(self.z_um,
                                       self.a_b_c_model_params['a'], 
                                       self.a_b_c_model_params['b'],
                                       self.a_b_c_model_params['c'],
                                      )

    def _calc_normalized_dose(self):        
        """Calculate normalized dose as a function of z."""
        norm_dose = np.zeros(len(self.z_um))
        for i, z_value in enumerate(self.z_um):
            norm_dose[i] = integrate.simps(
                self.irradiance(z_value), 
                self.irradiance.wavelengths
            ) / self.source_integral
        return norm_dose

    def _fit_to_ha_model(self):
        popt, pcov = fit_ha_model(self.z_um, self.normalized_dose)
        ha_model_params = {}
        ha_model_params['ha'] = popt[0]
        ha_model_params['stdev'] = np.sqrt(np.diag(pcov))
        return ha_model_params

    def _fit_to_a_b_model(self):
        popt, pcov = fit_a_b_model(self.z_um, self.normalized_dose)
        a_b_model_params = {}
        a_b_model_params['a'] = popt[0]
        a_b_model_params['b'] = popt[1]
        a_b_model_params['stdev'] = np.sqrt(np.diag(pcov))
        return a_b_model_params

    def _fit_to_a_b_c_model(self):
        popt, pcov = fit_a_b_c_model(self.z_um, self.normalized_dose)
        a_b_c_model = {}
        a_b_c_model['a'] = popt[0]
        a_b_c_model['b'] = popt[1]
        a_b_c_model['c'] = popt[2]
        a_b_c_model['stdev'] = np.sqrt(np.diag(pcov))
        return a_b_c_model

    def plot(self, grid=True, title=None, file=None):
        fig, ax = plt.subplots()
        ax.plot(self.z_um, self.normalized_dose, 'r', label="$D_n(z)$")
        
        text_func3 = 'fit: y = a*exp(-z/b) + c\n'
        model = self.a_b_c_model_params
        ax.plot(self.z_um, self.a_b_c_model, 'r--', 
                label=text_func3 + 'a={:4.3}, b={:5.3f}, c={:5.3f}'.format(model['a'], model['b'], model['c']))
        
        text_func2 = 'fit: y = a*exp(-z/b) + (1-a)\n'
        model = self.a_b_model_params
        ax.plot(self.z_um, self.a_b_model, 'g-.', 
                label=text_func2 + 'a={:4.3}, b={:5.3f}'.format(model['a'], model['b']))
        
        text_func1 = 'fit: y = exp(-z/b)\n'
        model = self.ha_model_params
        ax.plot(self.z_um, self.ha_model, 'r:', 
                label=text_func1 + 'b={:7.5f}'.format(model['ha']))

        ax.set_xlim(0, )
        ax.set_xlabel('z ($\mu$m)')
        ax.set_ylabel("D'(z)")
        ax.set_ylim(0,1)
        ax.legend()
        
        if grid:
            ax.grid()
        
        if title:
            ax.set_title(title)
        
        if file:
            plt.savefig(file)
            
        return fig, ax        

In [7]:
z_array_params = ArrayParams(0, 100, 201)

# Create resin
pegda = ResinConstituentMonomer(data.monomers['PEGDA'], 100)
avobenzone = ResinConstituentAbsorber(
    data.absorbers['avobenzone_in_PEGDA_2017-04-13'],
    0.38
)
irgacure819 = ResinConstituentAbsorber(
    data.photoinitiators['irgacure819_in_PEGDA_2017-04-13'],
    1.0
)
resin = Resin(monomers=pegda, absorbers=avobenzone, photoinitiators=irgacure819)

# Source
source = data.sources['HR3.2_365nm_370nm_short_pass_filter-2019-12-31']

# Irradiance
irradiance = Irradiance(source=source, resin=resin, wavelength_array_params=ArrayParams(300, 440, 281))

# Normalized dose
norm_dose = NormalizedDose(irradiance=irradiance, z_array_params=z_array_params)

print(norm_dose.irradiance.resin.name)

norm_dose.plot();

PEG-100__Avo-0.38__Irg-1


Canvas(toolbar=Toolbar(toolitems=[('Home', 'Reset original view', 'home', 'home'), ('Back', 'Back to previous …

In [8]:
avobenzone_2020 = ResinConstituentAbsorber(
    data.absorbers['avobenzone_in_PEGDA_2020-01-20'],
    0.38
)

resin_avo_2020 = Resin(monomers=pegda, absorbers=avobenzone_2020, photoinitiators=irgacure819)

irradiance_avo_2020 = Irradiance(source=source, resin=resin_avo_2020, wavelength_array_params=ArrayParams(300, 440, 281))

# Normalized dose
norm_dose_avo_2020 = NormalizedDose(irradiance=irradiance_avo_2020, z_array_params=z_array_params)

print(norm_dose_avo_2020.irradiance.source.name)

norm_dose_avo_2020.plot();


HR3.2_365nm_370nm_short_pass_filter-2019-12-31


Canvas(toolbar=Toolbar(toolitems=[('Home', 'Reset original view', 'home', 'home'), ('Back', 'Back to previous …

In [13]:
data.absorbers

{'avobenzone_in_PEGDA_2017-04-13': <spectra.absorberspectrum.AbsorberSpectrum at 0x7f915a803b10>,
 'avobenzone_in_PEGDA_2020-01-20': <spectra.absorberspectrum.AbsorberSpectrum at 0x7f915a65dc90>,
 'benetex_obplus_in PEGDA_2017-04-13': <spectra.absorberspectrum.AbsorberSpectrum at 0x7f9159c5b9d0>,
 'bls99-2_in_PEGDA_2017-04-13': <spectra.absorberspectrum.AbsorberSpectrum at 0x7f915a803a10>,
 'ideal_uniform_absorber': <spectra.absorberspectrum.AbsorberSpectrum at 0x7f915a80b150>,
 'martius_yellow_in PEGDA_2020-04-13': <spectra.absorberspectrum.AbsorberSpectrum at 0x7f915a817f90>,
 'nps_in_PEGDA_2017-04-13': <spectra.absorberspectrum.AbsorberSpectrum at 0x7f915a65dd50>,
 'octocrylene_in_PEGDA_2017-04-13': <spectra.absorberspectrum.AbsorberSpectrum at 0x7f915a803d50>,
 'phenazine_in_PEGDA_2017-04-13': <spectra.absorberspectrum.AbsorberSpectrum at 0x7f915a81f190>,
 'salicylaldehyde_in_PEGDA_2017-04-13': <spectra.absorberspectrum.AbsorberSpectrum at 0x7f915a81f650>,
 'sudanI_in_PEGDA_2017-04